# MediaForge on a free GPU (Colab / Kaggle)

Runs the **whole MediaForge backend on a free T4 GPU** and opens a public URL
you paste into MediaForge's frontend (or open directly). Everything — LTX-Video
motion, talking avatars, TTS — runs here for **free**, no API keys.

**Runtime → Change runtime type → GPU (T4)** before running.

Free-tier reality: ~5s per LTX segment, sessions time out after a few hours,
and the big Wan-14B model won't fit (use LTX or Wan-1.3B).

## 1 · Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2 · Get the code + install deps

In [ ]:
# Public repo — clones with no auth needed.
!git clone https://github.com/bigmo6286/mediaforge.git
import os
BACKEND = '/content/mediaforge/backend'
assert os.path.isdir(BACKEND), f'Clone failed? expected {BACKEND}'
%cd $BACKEND

In [ ]:
# Core API + local model stack (torch is preinstalled on Colab)
!pip install -q fastapi 'uvicorn[standard]' python-multipart httpx imageio-ffmpeg Pillow
!pip install -q diffusers transformers accelerate sentencepiece
# Talking avatar + TTS run through diffusers / hosted; SadTalker can be added here.
!pip install -q pyngrok

## 3 · Configure to run models locally on this GPU
No API keys — everything runs on the Colab GPU.

In [ ]:
import os
os.environ['MOTION_MODEL'] = 'ltx'      # efficient long-clip model
os.environ['MOTION_PROVIDER'] = 'local' # run on THIS gpu
os.environ['WAN_PROVIDER'] = 'local'
os.environ['AVATAR_PROVIDER'] = 'local'
os.environ['LTX_LOCAL_MODEL'] = 'Lightricks/LTX-Video'
os.environ['SEGMENT_SECONDS'] = '5'

## 4 · Start the backend + a public tunnel
Paste the printed URL into MediaForge's frontend (set it as the API base), or
open `<url>/docs` to drive the API directly.

In [ ]:
import subprocess, time, threading
from pyngrok import ngrok
# ngrok free needs a token: https://dashboard.ngrok.com/get-started/your-authtoken
# !ngrok config add-authtoken YOUR_TOKEN
proc = subprocess.Popen(['uvicorn', 'app.main:app', '--host', '0.0.0.0', '--port', '8000'])
time.sleep(6)
public = ngrok.connect(8000)
print('MediaForge API is live at:', public.public_url)
print('Interactive API docs   :', public.public_url + '/docs')

## 5 · Quick test — generate a 15s LTX clip
First call downloads the model weights (a few minutes).

In [ ]:
import httpx, time
base = public.public_url
r = httpx.post(base + '/api/generate/t2v', data={
    'prompt': 'a golden retriever puppy running through a sunny meadow, cinematic',
    'model': 'ltx', 'target_seconds': 15}, timeout=60)
job = r.json()['job_id']; print('job', job)
while True:
    s = httpx.get(f'{base}/api/jobs/{job}').json()
    print(s['status'], round(s['progress']*100), s['message'])
    if s['status'] in ('done', 'error'): break
    time.sleep(4)
print(s.get('result') or s.get('error'))